In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import pandas as pd 
from torch.utils.data import random_split 
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pyarrow.dataset as ds




In [2]:
local_model_path = "models/Qwen/Qwen3-1.7B"

model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    local_files_only=True
)

# load your trained LoRA
model = PeftModel.from_pretrained(model, "./3fr_en_adapter_best")

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [3]:

df = pd.read_csv('French_eng.csv')


train_texts, val_texts = train_test_split(
    df["data_generation_result"].tolist(),
    test_size=0.1,
    random_state=42
)

texts = val_texts

len(train_texts), len(val_texts)


(7263, 807)

In [4]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=400
)

In [5]:
labels = []
for seq in encodings["input_ids"]:
    labels.append([
        token if token != tokenizer.pad_token_id else -100
        for token in seq
    ])

encodings["labels"] = labels

In [2]:
class TextDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.input_ids[idx]),  
        }
    
#dataset = TextDataset(encodings)
# loader = DataLoader(dataset, batch_size=8)

In [7]:
import math

device = model.device
losses = []

with torch.no_grad():
    for batch in loader:
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device)
        )
        losses.append(outputs.loss.item())

ppl = math.exp(sum(losses) / len(losses))
print("Perplexity:", ppl)

Perplexity: 1.489396433899299


In [26]:
def generate(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

generate("Je veux")

"Je veux créer une application web pour suivre mes activités de santé. What are the steps I should take first? Also, how can I ensure that my data is secure?\n\nJe veux utiliser React pour construire l'interface de l'application. Are"

In [3]:
local_model_path = "models/Qwen/Qwen3-1.7B"

model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    local_files_only=True
)

model2 = PeftModel.from_pretrained(model, "./zh_en_adapter_best")
model2.eval()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fea

In [4]:
dataset = ds.dataset("./Chinese_eng" , format="parquet")
table = dataset.to_table()
df2 = table.to_pandas()

In [5]:
train_texts, val_texts = train_test_split(
    df2["transcription"].tolist(),
    test_size=0.1,
    random_state=42
)

texts = val_texts

len(train_texts), len(val_texts)

(13500, 1500)

In [6]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=200
)

In [7]:
labels = []
for seq in encodings["input_ids"]:
    labels.append([
        token if token != tokenizer.pad_token_id else -100
        for token in seq
    ])

encodings["labels"] = labels
dataset2 = TextDataset(encodings)
loader2 = DataLoader(dataset2, batch_size=8)

In [8]:
import math

device = model.device
losses = []

with torch.no_grad():
    for batch in loader2:
        outputs = model2(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device)
        )
        losses.append(outputs.loss.item())

ppl = math.exp(sum(losses) / len(losses))
print("Perplexity:", ppl)

Perplexity: 2.4175191853632056
